## Hit calling across all cell lines
This notebook `04_hit_calling_all.ipynb` does hit calling, i.e. ranks miRNAs by the
effect they had on cellular morphology, across all cell lines.

Before getting started with this notebook, make sure you have followed the setup
instructions in the [README.md](README.md).

Note: to make this notebook easier to run on devices without large RAM, you may
enable AnnData's `backed` mode to load in data only when required. This does make
processing slower. If you wish to use this mode, change `use_backed` to `True`
in the first code cell.

In [ ]:
import anndata as ad
import pickle
import pandas as pd
import scmorph as sm

file = "workspace/profiles_assembled/profiles_processed/all_adata_featFilt_batchCorr.h5ad"

# Save memory
use_backed = False

adata = sm.read(file, backed="r" if use_backed else False)

In [ ]:
# load dependencies
import pandas as pd
import numpy as np
import scmorph as sm
from tqdm import tqdm
from plotnine import *
import matplotlib.pyplot as plt
from natsort import natsorted
from processing_helper import map_cell_lines

In [3]:
adatas_per_plate = {}
for plate in adata.obs["PlateID"].unique():
    adata_ss = adata[adata.obs["PlateID"] == plate]
    adatas_per_plate[plate] = adata_ss

In [4]:
# Define control wells present across all plates
ctrl_wells = ['E23', 'E24', 'F23', 'F24', 'I23', 'I24', 'J23', 'J24']

# Test treatments' significance per plate
ref_ks_d = {}
treat_ks_d = {}
for plate in tqdm(adatas_per_plate):
    ref_ks_d[plate] = {}
    treat_ks_d[plate] = {}
    adata_ss = adatas_per_plate[plate]
    if adata_ss.isbacked:
        adata_ss = adata_ss.to_memory()
    ref_ks_d[plate], treat_ks_d[plate] = sm.tl.get_ks(adata_ss,
                                                        treatment_key="treatment",
                                                        control_wells=ctrl_wells,
                                                        progress=False,
                                                        n_pcs=10,
                                                        scale_by_var=True)

100%|██████████| 134/134 [07:47<00:00,  3.49s/it]


In [18]:
plateid_to_meta = adata.obs.set_index('PlateID')[["PlateLayout",'CellLine', 'Replicate']].drop_duplicates().apply(list, axis=1).to_dict()

In [ ]:
for plate in treat_ks_d:
    plate_layout, cell_line, rep = plateid_to_meta[plate]
    ref_ks_d[plate].insert(0, "PlateLayout", plate_layout)
    ref_ks_d[plate].insert(0, "Replicate", rep)
    ref_ks_d[plate].insert(0, "CellLine", cell_line)
    ref_ks_d[plate].insert(0, "PlateID", plate)
    treat_ks_d[plate].insert(0, "PlateLayout", plate_layout)
    treat_ks_d[plate].insert(0, "Replicate", rep)
    treat_ks_d[plate].insert(0, "CellLine", cell_line)
    treat_ks_d[plate].insert(0, "PlateID", plate)

In [ ]:
ref_ks=pd.concat(ref_ks_d).dropna()
treat_ks=pd.concat(treat_ks_d).dropna()
map_cell_lines(ref_ks)
map_cell_lines(treat_ks)
treat_ks.reset_index(drop=True).to_csv("output/all_hit_calling_results.csv",index=False)